# 🔬 MorphoBricks Cellpose

Segments the images of a described MorphoBricks preprocessing run with **Cellpose 4.2.1.1** and writes the masks where preprocessing expects them. Your images are never changed: masks go to a separate masks folder, with a `segmentation_record.json` that says how they were made.

### What it does

Open a run → choose the mask type and the channels **by channel ID** → choose the model and device → preview one image → segment all images → rerun preprocessing with the masks folder.

### Before you start

- A finished preprocessing run (run `MorphoBricks_Preprocessing` first, with the experiment described). The run knows each image's axes and channel IDs; nothing is guessed here.
- A masks folder outside the experiment folder and the run's output folder.
- **Google Colab:** keep the run and the masks folder on Google Drive, under **/content/drive/MyDrive/**, and choose a GPU runtime (Runtime > Change runtime type).
- **JupyterLab / LabConstrictor:** use normal local paths.

### Things to know

- **2D images only for now.** An image with more than one Z plane or time point fails with a message that says so; it is never reduced to one plane.
- **Channels by ID, at most three.** You choose channels by their ID (for example `DNA`, `ACTIN`). In a run where some images have a channel that others lack, only the channels every image has can be used; the notebook shows the refusal as it is. Cellpose reads at most three channels and drops the rest by position, so more than three are refused.
- **Models:** `cpsam_v2` (default), `cpsam`, `cpdino`, `cpdino-vitb`, or a model file of your own. A name or file Cellpose cannot find is **refused**; Cellpose itself would silently use `cpsam_v2` instead.
- **Downloads:** built-in weights are downloaded from Hugging Face (mouseland/cellpose-sam) on first use. Nothing downloads unless you tick the download box in cell 5. The `cpdino` models also need DINOv3 (installed only if you tick its box, at a pinned commit; see cell 5, which also says that the runtime must be restarted after that).
- **Device:** you choose `cuda`, `mps` or `cpu`. A GPU that is not available is **refused**; it never falls back to the CPU, and the notebook checks that every weight of the network sits on the device you chose (that is the network's placement only). Cellpose's own post-processing is not covered by that check: it may move its flow-threshold quality step to the CPU when GPU memory is short, and always computes the masks from the flows on the CPU on `mps`. That is Cellpose's own behaviour, it is **not refused**, and it is written to the record as a note.
- **Diameter:** in pixels. **Blank means the model's own native scale** (Cellpose 4.2.1.1 does not estimate a diameter). Zero is refused.
- Every setting that ran (the full normalisation and the flow iterations included), the model's file name and its SHA-256, and whether the weights were downloaded in this session and from where, are written to the record.

### How to run

Run the code cells in order (click a cell and press **Shift + Enter**, or its ▶ button). After a restart of the runtime, run cell 1 again: it installs nothing that is already there.

___

## **1. Install Cellpose and MorphoBricks preprocessing**

Installs Cellpose 4.2.1.1 and the preprocessing commit this notebook is made for. Nothing is installed when it is already there, so running the cell again is safe. It prints **Ready** with the versions. It never installs DINOv3: that is a choice in cell 5.

**Google Colab:** if the installation upgrades `numpy` or `opencv` while the runtime already has them loaded, the cell stops and tells you to restart the runtime (Runtime > Restart session). Do that, then run cell 1 again: it installs nothing more and prints **Ready**.

In [ ]:
# @title 1. Install Cellpose and MorphoBricks preprocessing
import subprocess
import sys


class CellStop(RuntimeError):
    """Stop this cell with its message on one line, without a traceback (Colab and Jupyter)."""

    def _render_traceback_(self) -> list:
        return [str(self)]


if sys.version_info < (3, 11):
    raise CellStop("Select a Python 3.11 or newer kernel, then run this cell again.")

import json
from importlib import invalidate_caches
from importlib.metadata import PackageNotFoundError, distribution, version

# LabConstrictor version identifiers.
current_version = "0.1.0"
notebook_name = "MorphoBricks_Cellpose"

# Both are pinned: Cellpose to the release this notebook is made for, preprocessing to the
# immutable source revision that holds this notebook's code.
CELLPOSE_VERSION = "4.2.1.1"
RELEASE_REF = "425ca3ef646c0c897820ce1343b6d607ae3e9544"
REPOSITORY = "https://github.com/CellMigrationLab/morphobricks-preprocessing.git"
EXTRAS = "notebook,zarr" if "google.colab" in sys.modules else "notebook"

# True only to develop against the preprocessing already loaded in this kernel (a local
# checkout): the cell then says so and installs no other preprocessing.
DEVELOPMENT_OVERRIDE = False
SETUP_RECORD: dict = {}
# Packages pip may upgrade under a running kernel; each maps to the module that stays loaded.
LOADED_BY_KERNEL = {"numpy": "numpy", "opencv-python": "cv2", "opencv-python-headless": "cv2"}


def _version(name: str) -> str | None:
    """Return the installed version of a package, or None."""
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def _installed_commit(name: str) -> str | None:
    """Return the Git commit a package was installed from (pip's direct_url.json), or None."""
    try:
        text = distribution(name).read_text("direct_url.json")
    except PackageNotFoundError:
        return None
    return json.loads(text).get("vcs_info", {}).get("commit_id") if text else None


def _install() -> None:
    """Install what is missing or differs from the pins, then report the versions."""
    wanted = []
    if _version("cellpose") != CELLPOSE_VERSION:
        wanted.append(f"cellpose=={CELLPOSE_VERSION}")
    preprocessing_ok = DEVELOPMENT_OVERRIDE or (
        _installed_commit("morphobricks-preprocessing") == RELEASE_REF
    )
    if not preprocessing_ok:
        wanted.append(f"morphobricks-preprocessing[{EXTRAS}] @ git+{REPOSITORY}@{RELEASE_REF}")
    if wanted:
        before = {name: _version(name) for name in LOADED_BY_KERNEL}
        print("Installing " + ", ".join(wanted) + "...", flush=True)
        try:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *wanted],
                check=True,
            )
        except (subprocess.CalledProcessError, OSError) as exc:
            raise CellStop(
                "The installation failed. Check the installation output and internet access, "
                "then run this cell again."
            ) from exc
        invalidate_caches()
        stale = [
            f"{name} {before[name]} -> {_version(name)}"
            for name, module in LOADED_BY_KERNEL.items()
            if before[name] and _version(name) != before[name] and module in sys.modules
        ]
        if stale:
            raise CellStop(
                f"The installation upgraded {', '.join(stale)} while the runtime already has it "
                "loaded. Restart the runtime (Colab: Runtime > Restart session), then run this "
                "cell again: it installs nothing more."
            )
    if _version("cellpose") != CELLPOSE_VERSION:
        raise CellStop(
            f"Cellpose {CELLPOSE_VERSION} is not installed ({_version('cellpose')}). Restart "
            "the runtime and run this cell again."
        )
    try:
        from morphobricks_preprocessing import notebook_cellpose  # noqa: F401
    except ImportError as exc:
        cause = " ".join(f"{type(exc).__name__}: {exc}".split())
        raise CellStop(f"MorphoBricks preprocessing could not load ({cause}).") from exc
    SETUP_RECORD.clear()
    SETUP_RECORD.update(
        notebook_version=current_version,
        preprocessing_version=_version("morphobricks-preprocessing"),
        cellpose_version=_version("cellpose"),
        release_ref=RELEASE_REF,
        development_override=DEVELOPMENT_OVERRIDE,
    )
    override = (
        " DEVELOPMENT OVERRIDE is on: preprocessing is not the pinned commit."
        if DEVELOPMENT_OVERRIDE
        else ""
    )
    print(
        f"Ready: Cellpose {_version('cellpose')}; preprocessing "
        f"{_version('morphobricks-preprocessing')} ({RELEASE_REF[:12]}).{override}\n"
        f"Notebook {current_version}.\n"
        "Next: run cell 2 (Mount Google Drive), then cell 3 (Open the run).",
        flush=True,
    )


_install()

___

## **2. Mount Google Drive**

**Google Colab:** connects your Google Drive. Allow access in the window that opens; your Drive is then the folder **/content/drive/MyDrive**. Keep the experiment and the output folder there: everything else on a Colab runtime is wiped when it disconnects.

**JupyterLab / LabConstrictor:** there is no Drive to mount. The cell only prints a note, and you use normal local paths in the app.

In [ ]:
# @title 2. Mount Google Drive
import sys


class CellStop(RuntimeError):
    """Stop this cell with its message on one line, without a traceback (Colab and Jupyter)."""

    def _render_traceback_(self) -> list:
        return [str(self)]


if "google.colab" in sys.modules:
    from google.colab import drive

    print("Google Colab: connecting Google Drive...", flush=True)
    try:
        drive.mount("/content/drive")
    except Exception as exc:
        # The cause stays on the one line: a stop must never hide what failed.
        cause = " ".join(f"{type(exc).__name__}: {exc}".split())
        raise CellStop(
            f"Google Drive could not be connected ({cause}). Allow Drive access, then run "
            "this cell again."
        ) from exc
    print("Google Drive is connected: your files are in /content/drive/MyDrive.", flush=True)
else:
    print("Not Google Colab: there is no Google Drive to mount here, so nothing was done.")

___

## **3. Open the preprocessing run**

Give the run's output folder and the masks folder. The notebook opens the run with preprocessing's own checks and lists the channels with the number of images that have each. Cell 3 stops with the reason when the run cannot be opened (a run that was stopped, or images changed since the run).

In [ ]:
# @title 3. Open the preprocessing run
# @markdown The run's **output folder** (it holds run_manifest.json) and the **masks
# @markdown folder** (outside the experiment and the run).
run_folder = ""  # @param {type:"string"}
masks_folder = ""  # @param {type:"string"}


class CellStop(RuntimeError):
    """Stop this cell with its message on one line, without a traceback."""

    def _render_traceback_(self) -> list:
        return [str(self)]


try:
    from morphobricks_preprocessing.notebook_cellpose import CellposeWorkflow
except ImportError as exc:
    raise CellStop("Run cell 1 first: it installs what this notebook needs.") from exc

workflow = CellposeWorkflow()
workflow.open(run_folder, masks_folder)

___

## **4. Choose the mask type and the channels**

The mask type is the name of the new masks (it becomes a folder in the masks folder and the name you declare in the experiment description). Choose one to three channels **by channel ID** in the dropdowns; their order is the order Cellpose gets them in. A channel that some images lack says so, and choosing it shows why it is refused.

In [ ]:
# @title 4. Choose the mask type and the channels
# @markdown The mask type to make (one folder name, for example Cells or Nuclei).
mask_type = "Cells"  # @param {type:"string"}


class CellStop(RuntimeError):
    """Stop this cell with its message on one line, without a traceback."""

    def _render_traceback_(self) -> list:
        return [str(self)]


if "workflow" not in globals():
    raise CellStop("Run cell 3 first.")
workflow.choose_channels(mask_type)

___

## **5. Choose the model and the settings, then load the model**

Built-in weights are downloaded from **Hugging Face** on first use, and only if you tick the download box. The notebook loads the model, then checks that the weights file is the one you asked for and that the network's weights are on the device you asked for. It shows the file's SHA-256, which is written to the record. The diameter is in pixels; blank is the model's native scale.

**The check covers every parameter and buffer of the network** (not Cellpose's own `device` attribute, which only repeats what it was given). Cellpose's post-processing is outside this check: it may move its flow-threshold quality step to the CPU when GPU memory is short, and always computes the masks from the flows on the CPU on `mps`. That is Cellpose's behaviour, not refused; the record carries it as a note, together with the full normalisation, the flow iterations, and whether the weights were downloaded now and from where.

**The `cpdino` models need DINOv3, and a restart.** Ticking the install box installs DINOv3 with `--no-deps` (so it cannot change your PyTorch) at commit `6876159a11b4`, **the head of its default branch on 22 November 2026, not verified against `cpdino`**. Cellpose imports DINOv3 only when it is first imported, so after installing DINOv3 **restart the runtime** (Colab: Runtime > Restart session), run cells 1 to 4 again and load a `cpdino` model; loading before the restart is refused with that instruction.

In [ ]:
# @title 5. Choose the model and the settings, then load the model
# @markdown ### Model and device
model = "cpsam_v2"  # @param ["cpsam_v2", "cpsam", "cpdino", "cpdino-vitb"]
# @markdown A model file of your own (leave blank to use the model above). A file
# @markdown that does not exist is refused.
custom_model_file = ""  # @param {type:"string"}
# @markdown The device is never chosen for you: a GPU that is not available is
# @markdown refused, not replaced by the CPU.
device = "cuda"  # @param ["cuda", "mps", "cpu"]
# @markdown Built-in weights are downloaded from Hugging Face on first use. Tick to
# @markdown allow that download.
download_weights = False  # @param {type:"boolean"}
# @markdown The cpdino models need DINOv3 (installed at a pinned commit). Tick to install it now.
install_dinov3 = False  # @param {type:"boolean"}
# @markdown ### Settings
# @markdown Diameter in pixels. Blank = the model's native scale (Cellpose 4.2.1.1
# @markdown does not estimate it). Never 0.
diameter = ""  # @param {type:"string"}
flow_threshold = 0.4  # @param {type:"number"}
cellprob_threshold = 0.0  # @param {type:"number"}
# @markdown Cellpose's own normalisation (each channel to its 1st-99th percentile).
normalize = True  # @param {type:"boolean"}
batch_size = 8  # @param {type:"integer"}


class CellStop(RuntimeError):
    """Stop this cell with its message on one line, without a traceback."""

    def _render_traceback_(self) -> list:
        return [str(self)]


if "workflow" not in globals():
    raise CellStop("Run cell 3 first.")
workflow.set_settings(diameter, flow_threshold, cellprob_threshold, normalize, batch_size)
workflow.load(model, custom_model_file, device, download_weights, install_dinov3)

___

## **6. Preview one image**

Segments one image and shows each channel (contrast stretched for display only; the masks and the stored values are never changed) and the outlines Cellpose found. Nothing is written. Change the settings in cell 5 and load again until it looks right.

In [ ]:
# @title 6. Preview one image
# @markdown The image to preview, as its path in the run (blank = the first image
# @markdown without a mask). Nothing is written.
preview_image = ""  # @param {type:"string"}


class CellStop(RuntimeError):
    """Stop this cell with its message on one line, without a traceback."""

    def _render_traceback_(self) -> list:
        return [str(self)]


if "workflow" not in globals():
    raise CellStop("Run cell 3 first.")
workflow.preview(preview_image)

___

## **7. Segment all images**

Segments every image that has no mask of this type yet and writes each mask as a 2D integer image in the masks folder, then writes the record. A progress line shows how far it is. The table at the end lists **every** image as written, skipped (its mask matches its record), needs attention (a mask with no record entry or other bytes; not replaced) or failed with the error: nothing is skipped silently, and one failed image does not stop the others. If a mask's record entry cannot be written, that mask is rolled back (an earlier mask restored) and the batch stops.

In [ ]:
# @title 7. Segment all images
# @markdown Tick to replace masks that already exist. Otherwise images that already
# @markdown have a mask are skipped.
overwrite = False  # @param {type:"boolean"}


class CellStop(RuntimeError):
    """Stop this cell with its message on one line, without a traceback."""

    def _render_traceback_(self) -> list:
        return [str(self)]


if "workflow" not in globals():
    raise CellStop("Run cell 3 first.")
result = workflow.run(overwrite)

___

## **8. What to do next**

1. Declare the new mask type in the experiment description (preprocessing notebook, step 4, or `morphobricks.toml`): for example `[masks.Cells]` with `source_type = "Cells"` and `representation = "instances"`.
2. Save the description.
3. Run preprocessing again with the masks folder (the histograms are reused).
4. Describe the masks.
5. Then continue with MorphoFeature.

In [ ]:
# @title 8. What to do next


class CellStop(RuntimeError):
    """Stop this cell with its message on one line, without a traceback."""

    def _render_traceback_(self) -> list:
        return [str(self)]


if "workflow" not in globals():
    raise CellStop("Run cell 3 first.")
workflow.next_steps()